# **Procesamiento de Lenguaje Natural**

## Maestría en Inteligencia Artificial Aplicada
#### Tecnológico de Monterrey
#### Prof Luis Eduardo Falcón Morales

### **Actividad en Equipos: sistema LLM + RAG con información tabular**

* **Nombres y matrículas:**




* A01769076 | Alonso Pedrero Martínez
* A01797612 | Lilian Margarita Álvarez Iglesias
* A01235844 | Luis Felipe Neri Alvarado Fregoso





* ##### **El formato de este cuaderno de Jupyter es libre, pero incluye al menos lo solicitado en el archivo PDF asociado a esta actividad.**

* ##### **Pueden importar los paquetes o librerías que requieran.**

* ##### **Pueden incluir las celdas y líneas de código que deseen.**

In [2]:
# Incluyan a continuación todas las celdas (de código o texto) que deseen...

from pypdf import PdfReader
import os
from unstructured.partition.pdf import partition_pdf
import nltk
import re
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss
from openai import OpenAI


/Users/alonsopedreromartinez/Documents/GitHub/llms-equipo14/activity5/test.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
load_dotenv()

True

In [4]:
nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/alonsopedreromartinez/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/alonsopedreromartinez/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## Extracting and cleaning the pdf information

In [5]:
PDF_PATH = "../files"
PYTHON_CHEATSHEET_FILENAME = "python_cheatsheet.pdf"
ML_CHEATSHEET_FILENAME = "ml_cheatsheet.pdf"

In [6]:
python_cheatsheet = os.path.join(PDF_PATH, PYTHON_CHEATSHEET_FILENAME)
ml_cheatsheet = os.path.join(PDF_PATH, ML_CHEATSHEET_FILENAME)

In [7]:
python_cheatsheet_text = ""
reader = PdfReader(python_cheatsheet)

for page in reader.pages:
    python_cheatsheet_text += page.extract_text()

print(python_cheatsheet_text)

Real PythonPocket Reference
Visit realpython.com to
turbocharge your
Python learning with
in-depth tutorials,
real-world examples,
and expert guidance.
Getting Started
Follow these guides to kickstart your Python journey:
realpython.com/what-can-i-do-with-python
realpython.com/installing-python
realpython.com/python-first-steps
Start the Interactive Shell
$ python
Quit the Interactive Shell
>>> exit()
Run a Script
$ python my_script.py
Run a Script in Interactive Mode
$ python -i my_script.py
Learn More on realpython.com/search:
interpreter ∙ run a script ∙ command line
Comments
Always add a space after the #
Use comments to explain “why” of your code
Write Comments
# This is a comment
# print("This code will not run.")
print("This will run.")  # Comments are ignored by Python
Learn More on realpython.com/search:
comment ∙ documentation
Data Types
Python is dynamically typed
Use None to represent missing or optional values
Use type() to check object type
Check for a specific type with 

In [8]:
elements = partition_pdf(
    ml_cheatsheet,
    languages=['eng'],
    strategy="hi_res",
    infer_table_structure=True,
    chunking_strategy="by_title",
    max_characters=1200,
    combine_text_under_n_chars=300
)

table_chunks = []
other_chunks = []

for e in elements:
    if type(e).__name__ == "TableChunk":
        table_chunks.append(e.text)
    else:
        other_chunks.append(e.text)

The `max_size` parameter is deprecated and will be removed in v4.26. Please specify in `size['longest_edge'] instead`.


In [9]:
elements_ocr = partition_pdf(
    filename=ml_cheatsheet,
    strategy="ocr_only"
)

all_text = [e.text for e in elements] + [e.text for e in elements_ocr]

In [10]:
def clean_np_str(text):
    return re.sub(r"np\.str_\('([^']*)'\)", r"\1", text)

In [11]:
clean_text = []
for i in range(len(all_text)):           
    clean_text.append(clean_np_str(all_text[i]))


In [12]:
algorithms = [
    "Linear Regression",
    "Logistic Regression",
    "Ridge Regression",
    "Lasso Regression",
    "Decision Tree",
    "Random Forests",
    "Gradient Boosting Regression",
    "XGBoost",
    "LightGBM Regressor",
    "K-Means",
    "Hierarchical Clustering",
    "Gaussian Mixture Models",
    "Apriori algorithm"
]

info_headers = ["DESCRIPTION", "APPLICATIONS", "ADVANTAGES", "DISADVANTAGES"]

In [ ]:
sections = {header: [] for header in info_headers}
current_header = None
current_block = []

for item in clean_text:
    if item in info_headers:
        if current_header and current_block:
            sections[current_header].append(current_block)

        current_header = item
        current_block = []
    else:
        if current_header:
            current_block.append(item)

# guardar último bloque
if current_header and current_block:
    sections[current_header].append(current_block)

print(sections)

{'DESCRIPTION': [['A simple algorithm that models a linear relationship between inputs and a continuous numerical output variable', 'A simple algorithm that models a linear relationship between inputs and a categorical output (1 or O)', 'Part of the regression family — it penalizes features that have low predictive outcomes by shrinking their coefficients closer to zero. Can be used for classification or regression', 'Part of the regression family — it penalizes features that have low predictive outcomes by shrinking their coefficients to zero. Can be used for classification or regression', 'Decision Tree models make decision rules on the features to produce predictions. It can be used for classification or regression', 'An ensemble learning method that combines the output of multiple decision trees', 'Gradient Boosting Regression employs boosting to make predictive models from an ensemble of weak predictive learners', 'Gradient Boosting algorithm that is efficient & flexible. Can be u

In [ ]:
applications = sections["APPLICATIONS"][0]

grouped_apps = []

for item in applications:
    item = item.strip()

    item = item.replace("USE CASES", "").strip()

    grouped_apps.append([item])

grouped_clean_applications = []
current_group = []
last_num = 0

for block in grouped_apps:
    for item in block:
        if not item.strip():
            continue

        matches = re.findall(r'(\d+)\.\s*[^0-9]+', item)

        parts = re.split(r'(?=\d+\.\s)', item)

        for p in parts:
            p = p.strip()
            if not p:
                continue

            num = int(re.match(r'(\d+)\.', p).group(1))

            if num < last_num:
                grouped_clean_applications.append(current_group)
                current_group = []

            current_group.append(p)
            last_num = num

if current_group:
    grouped_clean_applications.append(current_group)

print(grouped_clean_applications)

[['1. Stock price prediction', '2. Predicting housing prices', '3. Predicting customer lifetime value'], ['1. Credit risk score prediction', '2. Customer churn prediction'], ['1. Predictive maintenance for automobiles', '2. Sales revenue prediction'], ['1. Predicting housing prices', '2. Predicting clinical outcomes based on health data'], ['1. Customer churn prediction', '2. Credit score modeling', '3. Disease prediction'], ['1. Credit score modeling', '2. Predicting housing prices'], ['1. Predicting car emissions', '2. Predicting ride hailing fare amount'], ['1. Churn prediction', '2. Claims processing in insurance'], ['1. Predicting flight time for airlines', '2. Predicting cholesterol levels based on health data'], ['1. Customer segmentation', '2. Recommendation systems'], ['1. Fraud detection', '2. Document clustering based on similarity'], ['1. Customer segmentation', '2. Recommendation systems'], ['1. Product placements', '2. Recommendation engines', '3. Promotion optimization']

In [15]:
grouped_clean_descriptions = []

for desc in sections["DESCRIPTION"][0]:
    grouped_clean_descriptions.append([desc])

print(grouped_clean_descriptions)

[['A simple algorithm that models a linear relationship between inputs and a continuous numerical output variable'], ['A simple algorithm that models a linear relationship between inputs and a categorical output (1 or O)'], ['Part of the regression family — it penalizes features that have low predictive outcomes by shrinking their coefficients closer to zero. Can be used for classification or regression'], ['Part of the regression family — it penalizes features that have low predictive outcomes by shrinking their coefficients to zero. Can be used for classification or regression'], ['Decision Tree models make decision rules on the features to produce predictions. It can be used for classification or regression'], ['An ensemble learning method that combines the output of multiple decision trees'], ['Gradient Boosting Regression employs boosting to make predictive models from an ensemble of weak predictive learners'], ['Gradient Boosting algorithm that is efficient & flexible. Can be use

In [16]:
grouped_clean_advantages = []
current_group = []
last_num = 0

for item in sections["ADVANTAGES"][0]:
    if not item.strip():
        continue

    parts = re.split(r'(?=\d+\.\s)', item)

    for p in parts:
        p = p.strip()
        if not p:
            continue

        num = int(re.match(r'(\d+)\.', p).group(1))

        if num < last_num:
            grouped_clean_advantages.append(current_group)
            current_group = []

        current_group.append(p)
        last_num = num

if current_group:
    grouped_clean_advantages.append(current_group)

print(grouped_clean_advantages)

[['1. Explainable method', '2. Interpretable results by its output coefficients', '3. Faster to train than other machine learning models'], ['1. Interpretable and explainable', '2. Less prone to overfitting when using regularization', '3. Applicable for multi-class predictions'], ['1. Less prone to overfitting', '2. Best suited where data suffer from multicollinearity', '3. Explainable & interpretable'], ['1. Less prone to overfitting', '2. Can handle high-dimensional data', '3. No need for feature selection'], ['1. Explainable and interpretable', '2. Can handle missing values'], ['1. Reduces overfitting', '2. Higher accuracy compared to other models'], ['1. Better accuracy compared to other regression models', '2. It can handle multicollinearity', '3. It can handle non-linear relationships'], ['1. Provides accurate results', '2. Captures non linear relationships'], ['1. Can handle large amounts of data', '2. Computational efficient & fast training speed', '3. Low memory usage'], ['1. 

In [17]:
grouped_clean_disadvantages = []
current_group = []
last_num = 0

for item in sections["DISADVANTAGES"][0]:
    if not item.strip():
        continue

    parts = re.split(r'(?=\d+\.\s)', item)

    for p in parts:
        p = p.strip()
        if not p:
            continue

        num = int(re.match(r'(\d+)\.', p).group(1))

        # cortar si reinicia O repite número
        if num <= last_num:
            grouped_clean_disadvantages.append(current_group)
            current_group = []

        current_group.append(p)
        last_num = num

if current_group:
    grouped_clean_disadvantages.append(current_group)

print(grouped_clean_disadvantages)

[['1. Assumes linearity between inputs and output', '2. Sensitive to outliers', '3. Can underfit with small, high-dimensional data'], ['1. Assumes linearity between inputs and outputs', '2. Can overfit with small, high-dimensional data'], ['1. All the predictors are kept in the final model', "2. Doesn't perform feature selection"], ['1. Can lead to poor interpretability as it can keep highly correlated variables'], ['1. Prone to overfitting', '2. Sensitive to outliers'], ['1. Training complexity can be high', '2. Not very interpretable'], ['1. Sensitive to outliers and can therefore cause overfitting', '2. Computationally expensive and has high complexity'], ['1. Hyperparameter tuning can be complex', '2. Does not perform well on sparse datasets'], ['1. Can overfit due to leaf-wise splitting and high sensitivity', '2. Hyperparameter tuning can be complex'], ['1. Requires the expected number of clusters from the beginning', '2. Has troubles with varying cluster sizes and densities'], ["

In [18]:
ml_cheatsheet_text = ""
for i in range(len(algorithms)):
    #algorithms
    ml_cheatsheet_text += "ALGORITHM "
    ml_cheatsheet_text += (algorithms[i] + " ")
    #descriptions
    ml_cheatsheet_text += (info_headers[0] + " ")
    ml_cheatsheet_text += (str(grouped_clean_descriptions[i]) + " ")
    #applications
    ml_cheatsheet_text += (info_headers[1] + " ")
    ml_cheatsheet_text += (str(grouped_clean_applications[i]) + " ")
    #advantages
    ml_cheatsheet_text += (info_headers[2] + " ")
    ml_cheatsheet_text += (str(grouped_clean_advantages[i]) + " ")
    #disadvantages
    ml_cheatsheet_text += (info_headers[3] + " ")
    ml_cheatsheet_text += (str(grouped_clean_disadvantages[i]) + " ")

print(ml_cheatsheet_text)

ALGORITHM Linear Regression DESCRIPTION ['A simple algorithm that models a linear relationship between inputs and a continuous numerical output variable'] APPLICATIONS ['1. Stock price prediction', '2. Predicting housing prices', '3. Predicting customer lifetime value'] ADVANTAGES ['1. Explainable method', '2. Interpretable results by its output coefficients', '3. Faster to train than other machine learning models'] DISADVANTAGES ['1. Assumes linearity between inputs and output', '2. Sensitive to outliers', '3. Can underfit with small, high-dimensional data'] ALGORITHM Logistic Regression DESCRIPTION ['A simple algorithm that models a linear relationship between inputs and a categorical output (1 or O)'] APPLICATIONS ['1. Credit risk score prediction', '2. Customer churn prediction'] ADVANTAGES ['1. Interpretable and explainable', '2. Less prone to overfitting when using regularization', '3. Applicable for multi-class predictions'] DISADVANTAGES ['1. Assumes linearity between inputs an

## Chunking and vector database

In [19]:
def chunk_text(text, chunk_size=500, overlap=100):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap

    return chunks

In [20]:
python_chunks = chunk_text(python_cheatsheet_text)
ml_chunks = chunk_text(ml_cheatsheet_text)

all_chunks = python_chunks + ml_chunks

In [21]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embedding_model.encode(all_chunks)

/Users/alonsopedreromartinez/Documents/GitHub/llms-equipo14/activity5/test.venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [22]:
embeddings = np.array(embeddings).astype("float32")

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

## Model Selection and RAG

In [27]:
API_KEY = os.getenv("OPEN_AI_KEY")

In [29]:
client = OpenAI(
    api_key=API_KEY
)

In [30]:
def embed_query(query):
    return embedding_model.encode([query]).astype("float32")

In [31]:
def retrieve(query, k=3):
    query_vec = embed_query(query)

    distances, indices = index.search(query_vec, k)

    results = [all_chunks[i] for i in indices[0]]
    return results

In [32]:
def build_prompt(query, context_chunks):
    context = "\n\n".join(context_chunks)

    prompt = f"""
    You are a helpful assistant. Use the context below to answer.

    Context:
    {context}

    Question:
    {query}

    Answer:
    """
    return prompt

In [33]:
def ask_llm(query):
    docs = retrieve(query, k=3)
    prompt = build_prompt(query, docs)

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    return response.choices[0].message.content

## QUESTIONS

- a. According to the 'Exceptions' section, what exception is thrown if I divide by zero?

- b. Can you provide two examples of string methods?

- c. What are the main disadvantages of using the Random Forest model?

- d. What are three use cases for unsupervised learning techniques in clustering?

- e. Include at least one more question about the Python sheet.

- f. Include at least one more question about the ML sheet.

In [34]:
question1 = "According to the 'Exceptions' section, what exception is thrown if I divide by zero?"
answer = ask_llm(question1)
print(answer)

In the 'Exceptions' section, a `ZeroDivisionError` is thrown if you divide by zero.


In [35]:
question2 = "Can you provide two examples of string methods?"
answer = ask_llm(question2)
print(answer)

Sure! Here are two examples of string methods:

1. The `upper()` method:
   ```python
   "hello".upper()  # "HELLO"
   ```

2. The `replace()` method:
   ```python
   "Hello World".replace("World", "Python")  # "Hello Python"
   ```


In [36]:
question3 = "What are the main disadvantages of using the Random Forest model?"
answer = ask_llm(question3)
print(answer)

The main disadvantages of using the Random Forest model are:

1. Training complexity can be high.
2. It can lead to poor interpretability as it can keep highly correlated variables.


In [37]:
question4 = "What are three use cases for unsupervised learning techniques in clustering?"
answer = ask_llm(question4)
print(answer)

Three use cases for unsupervised learning techniques in clustering are:

1. Customer segmentation - This involves grouping customers based on their behaviors or characteristics to tailor marketing strategies and improve customer experience.

2. Recommendation systems - Clustering can be used to group similar items or users, allowing for personalized recommendations based on user preferences or similar customer behaviors.

3. Fraud detection - Clustering techniques can help identify unusual patterns in transaction data, allowing for the detection of potentially fraudulent activities by grouping similar transactions.


In [38]:
question5 = "According to the augmented assignments section, what does the examples mean?"
answer = ask_llm(question5)
print(answer)

The examples in the augmented assignments section demonstrate shorthand ways to update the value of a variable based on its current value. 

1. `counter += 1` means that the value of `counter` is incremented by 1. This is equivalent to writing `counter = counter + 1`.

2. `numbers += [4, 5]` means that the list `numbers` is extended by adding the elements `[4, 5]` to it. This is equivalent to writing `numbers = numbers + [4, 5]`.

3. `permissions |= write` means that the `permissions` variable is updated with the result of a bitwise OR operation with `write`. This is equivalent to writing `permissions = permissions | write`.

These augmented assignment operators provide a more concise and often clearer way to modify variables.


In [45]:
question6 = "According to the Lasso regression, what are the advantages?"
answer = ask_llm(question6)
print(answer)

The advantages of Lasso regression are:

1. Less prone to overfitting.
2. Can handle high-dimensional data.
3. No need for feature selection.


# **Conclusiones:**

* #### **Incluyan sus conclusiones de la actividad chatbot LLM + RAG para documentos con información tabular:**


Esta actividad de RAG (Retrieval-Augmented Generation) representó uno de los mayores retos del proceso, principalmente en la etapa de extracción y preparación de datos desde archivos PDF. Si bien la extracción de texto es relativamente directa cuando el contenido es digital y seleccionable, el problema se vuelve significativamente más complejo cuando los documentos contienen imágenes o son escaneos. En estos casos, la calidad del PDF influye directamente en la calidad del texto extraído, lo cual impacta de forma crítica el rendimiento del sistema RAG.


Otro punto clave fue la normalización del texto, la cual requirió un enfoque adaptado al tipo de datos disponibles (tailor-made). Este enfoque fue adecuado para el alcance de la actividad, ya que permitió estructurar la información de manera funcional para su uso en embeddings y recuperación. Sin embargo, en un entorno de producción o a mayor escala, este proceso debería diseñarse de forma más generalizable y robusta para adaptarse a distintos tipos de documentos y formatos.


Finalmente, el uso de un modelo de lenguaje contribuyó a mejorar la calidad de las respuestas generadas, incluso en casos donde las consultas eran similares o repetitivas. En algunos escenarios, el modelo es capaz de inferir o complementar información basada en su entrenamiento, lo cual mejora la experiencia de búsqueda, aunque también introduce la necesidad de evaluar cuidadosamente la relevancia y precisión de las respuestas. Por ello, es fundamental seleccionar y ajustar los modelos considerando siempre la efectividad y el objetivo específico del sistema de recuperación.


Esta actividad permitió comprender que el rendimiento de un sistema RAG no depende únicamente del modelo, sino de toda la cadena de procesamiento: desde la calidad de los datos, la extracción, la normalización, hasta la estrategia de recuperación y generación.


# **Fin de la actividad chatbot: LLM + RAG**